# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Andrew417/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (Refresh / Content Opportunity Scoring)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a ranking (scoring) task. We're not asking a yes/no per page in isolation — we're asking how urgently each page needs a refresh compared to every other page.
Because the team can only act on 50 pages a month, a plain yes/no label wouldn't tell
us which 50 to pick first. We need every page scored and ordered, so we can take the
top 50 off the ranked list.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

I am predicting `is_declining_label`. This is a **proxy**, not an observed outcome —
it is computed from `trend_direction` (in turn derived from `trend_pct`), which only
looks at data already sitting in the current 90-day window. Nothing about it reaches
into a future time period, so it does not reflect an outcome that was actually
observed happening over time.

The real thing we care about — whether a page deserves one of the team's 50 refresh
slots — can't be measured directly, since no one has labeled "deserves a slot."
`is_declining_label` stands in for it because a declining page plausibly suggests
the content has gone stale relative to what currently ranks or engages readers, and
a refresh (rewrite/update) is one of the few direct levers a team has to address
that. This connection is correlational, not proven: decline can also come from
consolidation, seasonality, or SERP changes that a refresh would not fix — so the
proxy is reasonable, not guaranteed.

`trend_direction` and `trend_pct` are excluded from features, because they are the
same information the label is computed from — including them would let the model
just read the label back off a feature instead of learning any real pattern
(a circular result).

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@50 is the metric. Given a 50-page monthly review capacity, the team
only acts on the top 50 ranked pages — so accuracy or ROC-AUC (which judge the
whole 30,000-page list) don't match the actual decision. Precision@50 checks:
of the top 50 the model ranked highest, how many were genuinely declining. The
other 29,950 pages aren't ignored — they were already compared against every
other page during ranking, they just scored lower.

Target: beat the baseline rule's Precision@50 of 0.240.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

one row = one unique page

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df.head()
df['is_declining_label'] = df['trend_direction'] == 'down'
df['is_declining_label'].value_counts()

df['content_id'].nunique(), len(df)

(30000, 30000)

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like the baseline (0.40*visibility + 0.30*freshness_risk + 0.25*position_opportunity+ 0.05*depth_gap) applies the same weights to every page, every time. It can't express
conditions — for example, low CTR might only signal a real problem when position is already
weak, but be meaningless when position is strong. A weighted sum has no way to say "this
matters only when that other thing is also true" — it just adds fixed numbers regardless of
context.

A trained model instead learns these conditional patterns directly from 30,000 real examples,
rather than a human guessing which combinations matter and how much to weight them.

The evidence this isn't just theoretical: on the same data and the same 50-slot capacity, the
baseline rule scored 0.240 Precision@50 while the random forest scored 0.740 — roughly 3x
better. That gap is exactly what "extra complexity earning its keep" (per the lane guide) looks
like in practice, not assumed in advance.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.